Some modifications I am making to my volumeHT
1. Want to adjust volumeHT so that was can choose our coordinates for
the overall space not just the side length. Like we won't assume that
each side ranges from -L/2 to L/2
2. add the box's center as an object variable an then add the fromCenter
method.

In [1]:
import numpy
import boxes

In [2]:
class volumeHT:
    
    def __init__(self, spaceCoordinates, c, d, boxType):
        self.conc = c #concentration
        # spaceCoordinates should give the coordinates of the overall
        # volume in space. Should be in the form [xmin, xmax, ymin, ymax,
        # zmin, zmax]
        lengths = []
        for i in range(3):
            lengths.append(spaceCoordinates[2*i + 1] - spaceCoordinates[2*i])
        self.lengths = tuple(lengths) #save all dimensions for iterating
        tempS = numpy.cbrt(d/c) #compute the approximate box size using
        # the above formula (temp here is temporary not temperature)
        s = [] #save the actual box dimensions
        numberOfBoxes = []
        for length in self.lengths: #follow above formula to find
                                    # integer number of boxes in each dimension
            b = int(numpy.round(length/tempS))
            sDim = length/b #also find real box length
            numberOfBoxes.append(b) #save these values
            s.append(sDim)
        self.numberOfBoxesPerDim = tuple(numberOfBoxes) #save to object variable
        self.sideLengths = tuple(s)
        self.boxDict = {}
        for i in range(self.numberOfBoxesPerDim[0]):
            xcenter = spaceCoordinates[0] + i*self.sideLengths[0] + self.sideLengths[0]/2
            for j in range(self.numberOfBoxesPerDim[1]):
                ycenter = spaceCoordinates[1] + i*self.sideLengths[1] + self.sideLengths[1]/2
                for k in range(self.numberOfBoxesPerDim[2]):
                    zcenter = spaceCoordinates[2] + i*self.sideLengths[2] + self.sideLengths[2]/2
                    self.boxDict[(i, j, k)] = boxType((i, j, k), (xcenter, ycenter, zcenter), self.sideLengths)
    
    # find the box corresponding to the coordinates
    def coordsToBox(self, coords):
        boxCoords = []
        coords = coords + numpy.array([self.lengths[0]/2,
                                       self.lengths[1]/2, self.lengths[2]])
        for i in range(3):
            boxCoords.append(numpy.floor(coords[i]/self.sideLengths[i]))
        return numpy.array(boxCoords)
    
    #find the adjacent boxes
    def adjacent(self, box):
        lst = []
        for i in [-1, 0, 1]:
            new_x = box[0] + i
            if new_x < 0 or new_x >= self.numberOfBoxesPerDim[0]:
                continue
            for j in [-1, 0, 1]:
                new_y = box[1] + j
                if new_y < 0 or new_y >= self.numberOfBoxesPerDim[1]:
                    continue
                for k in [-1, 0, 1]:
                    new_z = box[2] + k
                    if new_z < 0 or new_z >= self.numberOfBoxesPerDim[2]:
                        continue
                    if (i, j, k) != (0, 0, 0):
                        lst.append((new_x, new_y, new_z))
        return lst

    # put a particle in the hash table
    def put(self, item, coords):
        box = self.coordsToBox(coords)
        self.boxDict[box].add(item)
    
    # deletes an item from the hash table
    def delete(self, item, coords):
        box = self.coordsToBox(coords)
        self.boxDict[box].delete(item)
    
    # get a list of all particles in the HashTable
    def contents(self):
        allItems = []
        allBoxes = self.boxDict.values()
        for box in allBoxes:
            for ob in box.contents:
                allItems.append(ob)
        return allItems

Now I need to test this code!

Boo!

I'm going to start by checking the volumeHT constructor

In [3]:
# I need to write a function that checks the constructor

# first I will try a very simple case. I have a cube of sidelength 3,
# going from 0 to 3 on each side, and my concentration and density are 1

volumeHTsimple = volumeHT([0, 3, 0, 3, 0, 3], 1, 1, boxes.overlapBox)
